# YOLO26m - Emergency Vehicle Detection
**Proyecto:** Getaway Car Identification and Tracking in Drone Surveillance  
**Hardware de entrenamiento:** Kaggle 2x T4 GPUs  
**Despliegue:** Jetson Orin Nano  
**Clases:** ambulance, fire_truck, police_car, other_vehicles

## 1. Instalacion de dependencias

In [ ]:
!pip install ultralytics roboflow albumentations pyyaml opencv-python-headless -q

## 2. Imports

In [ ]:
import os
import shutil
import random
import yaml
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
from ultralytics import YOLO
import albumentations as A

random.seed(42)
np.random.seed(42)

# Directorio de trabajo local
WORKING_DIR = Path.cwd() / "output"
WORKING_DIR.mkdir(parents=True, exist_ok=True)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print(f"Working dir: {WORKING_DIR}")

## 3. Descarga del dataset desde Roboflow

In [ ]:
import requests
import time
import urllib.request
from roboflow import Roboflow

API_KEY   = "nktAfMlO3madJ0g77qWg"
WORKSPACE = "darksoul98207s-workspace"
PROJECT   = "emergency_vehicles_mexico"
OUTPUT    = Path("/kaggle/working/emergency_vehicles_raw")

# Project info + class names
info = requests.get(f"https://api.roboflow.com/{WORKSPACE}/{PROJECT}?api_key={API_KEY}").json()
class_names = list(info["project"]["classes"].keys())
print(f"Classes ({len(class_names)}): {class_names}")

# Directory structure
for split in ["train", "valid", "test"]:
    (OUTPUT / split / "images").mkdir(parents=True, exist_ok=True)
    (OUTPUT / split / "labels").mkdir(parents=True, exist_ok=True)

# List all images via SDK search endpoint
rf      = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)

print("Fetching image list...")
all_images = []
for page in project.search_all(fields=["id", "name", "split"], limit=100):
    all_images.extend(page)
    print(f"  {len(all_images)} images fetched...")
print(f"Total: {len(all_images)} images")

# Download images and write YOLO labels
for i, img_meta in enumerate(all_images):
    img_id = img_meta["id"]
    split  = img_meta.get("split", "train")

    detail     = project.image(img_id)
    annotation = detail.get("annotation") or {}
    W          = annotation.get("width", 1)
    H          = annotation.get("height", 1)
    fname      = detail["name"]
    stem       = Path(fname).stem
    img_url    = detail["urls"]["original"]

    dest_img = OUTPUT / split / "images" / fname
    if not dest_img.exists():
        urllib.request.urlretrieve(img_url, dest_img)

    dest_lbl = OUTPUT / split / "labels" / f"{stem}.txt"
    lines = []
    for box in annotation.get("boxes", []):
        label = box.get("label", "")
        if label not in class_names:
            continue
        cls_id = class_names.index(label)
        cx = float(box["x"]) / W
        cy = float(box["y"]) / H
        nw = float(box["width"]) / W
        nh = float(box["height"]) / H
        lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

    dest_lbl.write_text("\n".join(lines))

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(all_images)} done")
    time.sleep(0.05)

# data.yaml
yaml_data = {
    "train": "../train/images",
    "val":   "../valid/images",
    "test":  "../test/images",
    "nc":    len(class_names),
    "names": class_names,
}
with open(OUTPUT / "data.yaml", "w") as f:
    yaml.dump(yaml_data, f, default_flow_style=False, allow_unicode=True)

DATASET_PATH = OUTPUT
print(f"Dataset descargado en: {DATASET_PATH}")
print("Estructura:")
for p in sorted(DATASET_PATH.rglob("*")):
    if p.is_dir():
        print(f"  {p.relative_to(DATASET_PATH)}/")


In [ ]:
import os
import json
import time
import urllib.request
from pathlib import Path
from collections import defaultdict

os.environ["ULTRALYTICS_API_KEY"] = "ul_c52eba4d9bf32c0450ec3a05b1a2ed29c6f6444b"

NDJSON_PATH = Path("lufess/datasets/ambulancias/ambulancias.ndjson")

if not NDJSON_PATH.exists():
    from ultralytics.data.utils import check_det_dataset
    try:
        check_det_dataset("ul://lufess/datasets/ambulancias")
    except AssertionError:
        pass  # NDJSON downloads before the YAML assertion fires

assert NDJSON_PATH.exists(), f"NDJSON not found at {NDJSON_PATH}"

# Class ID remapping: ambulancias → main dataset
# ambulancias: {0:ambulance, 1:other_vehicle, 2:police_car, 3:fire_truck}
# main:        {0:fire_truck, 1:ambulance,    2:police_car, 3:other_vehicle}
AMB_TO_MAIN  = {0: 1, 1: 3, 2: 2, 3: 0}
MAIN_NAMES   = {0: "fire_truck", 1: "ambulance", 2: "police_car", 3: "other_vehicle"}
SPLIT_MAP    = {"train": "train", "val": "valid", "valid": "valid", "test": "test"}
MAX_RETRIES  = 3

def download_with_retry(url, dst, retries=MAX_RETRIES):
    for attempt in range(retries):
        try:
            urllib.request.urlretrieve(url, dst)
            return True
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)  # 1s, 2s backoff
            else:
                return False

merged_total  = 0
skipped_total = 0
# per-class instance counts for what actually got merged
merged_cls    = defaultdict(int)
skipped_cls   = defaultdict(int)  # instances in skipped images

with open(NDJSON_PATH) as f:
    for i, line in enumerate(f):
        entry = json.loads(line)
        if entry.get("type") != "image":
            continue

        split  = SPLIT_MAP.get(entry.get("split", "train"), "train")
        url    = entry.get("url", "")
        fname  = entry.get("file", f"img_{i}.jpg")
        stem   = Path(fname).stem
        suffix = Path(fname).suffix or ".jpg"
        W      = entry.get("width",  1)
        H      = entry.get("height", 1)

        new_stem = f"amb_{stem}"
        dst_img  = DATASET_PATH / split / "images" / f"{new_stem}{suffix}"
        dst_lbl  = DATASET_PATH / split / "labels"  / f"{new_stem}.txt"

        annotations = entry.get("annotations") or {}
        boxes = (annotations.get("detect")
                 or annotations.get("boxes")
                 or annotations.get("bboxes")
                 or [])

        if not dst_img.exists():
            if not url:
                for box in boxes:
                    if len(box) >= 1:
                        skipped_cls[AMB_TO_MAIN.get(int(box[0]), int(box[0]))] += 1
                skipped_total += 1
                continue
            ok = download_with_retry(url, dst_img)
            if not ok:
                print(f"  FAIL (3 attempts): {fname}")
                for box in boxes:
                    if len(box) >= 1:
                        skipped_cls[AMB_TO_MAIN.get(int(box[0]), int(box[0]))] += 1
                skipped_total += 1
                continue

        label_lines = []
        for box in boxes:
            if len(box) < 5:
                continue
            cls_id = int(box[0])
            cx, cy, bw, bh = float(box[1]), float(box[2]), float(box[3]), float(box[4])
            if cx > 1 or cy > 1:
                cx /= W; cy /= H; bw /= W; bh /= H
            main_cls = AMB_TO_MAIN.get(cls_id, cls_id)
            label_lines.append(f"{main_cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
            merged_cls[main_cls] += 1

        dst_lbl.write_text("\n".join(label_lines))
        merged_total += 1

        if merged_total % 50 == 0:
            print(f"  {merged_total} images merged...")

print(f"\nDone. Merged: {merged_total}, skipped: {skipped_total}")
print("\nMerged instances per class:")
for cls_id, name in MAIN_NAMES.items():
    print(f"  {name}: {merged_cls[cls_id]}")
if skipped_total:
    print("\nSkipped instances per class (lost):")
    for cls_id, name in MAIN_NAMES.items():
        if skipped_cls[cls_id]:
            print(f"  {name}: {skipped_cls[cls_id]}")


## 3.5. Correccion de orientacion EXIF

In [ ]:
from PIL import Image, ImageOps

def fix_exif_orientation(dataset_path: Path):
    fixed = 0
    for split in ["train", "valid", "test"]:
        img_dir = dataset_path / split / "images"
        if not img_dir.exists():
            continue
        for img_path in list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.jpeg")) + list(img_dir.glob("*.png")):
            img = Image.open(img_path)
            corrected = ImageOps.exif_transpose(img)
            if corrected is not img:
                if corrected.mode not in ("RGB", "RGBA"):
                    corrected = corrected.convert("RGB")
                corrected.save(img_path)
                fixed += 1
    print(f"EXIF corregido en {fixed} imagenes.")

fix_exif_orientation(DATASET_PATH)

## 4. Analisis de estructura del dataset

In [ ]:
# Leer data.yaml
yaml_path = DATASET_PATH / "data.yaml"
with open(yaml_path) as f:
    data_cfg = yaml.safe_load(f)

CLASS_NAMES = data_cfg["names"]
NUM_CLASSES = data_cfg["nc"]
print(f"Clases ({NUM_CLASSES}): {CLASS_NAMES}")

# Contar imagenes por split
for split in ["train", "valid", "test"]:
    img_dir = DATASET_PATH / split / "images"
    if img_dir.exists():
        imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpeg"))
        print(f"  {split}: {len(imgs)} imagenes")

## 5. Analisis de distribucion de clases

In [ ]:
def analyze_dataset(split_dir: Path, class_names: list):
    """Analiza la distribucion de clases en un split del dataset.
    
    Retorna:
        instance_counts: Counter con instancias (anotaciones) por clase
        images_per_class: dict {class_id: [lista de rutas de imagenes]}
        background_images: lista de imagenes sin anotaciones
    """
    images_dir = split_dir / "images"
    labels_dir = split_dir / "labels"

    image_files = (
        list(images_dir.glob("*.jpg"))
        + list(images_dir.glob("*.png"))
        + list(images_dir.glob("*.jpeg"))
    )

    instance_counts = Counter()
    images_per_class = defaultdict(list)
    background_images = []

    for img_path in image_files:
        label_path = labels_dir / (img_path.stem + ".txt")

        if not label_path.exists():
            background_images.append(img_path)
            continue

        with open(label_path) as f:
            lines = [l.strip() for l in f.readlines() if l.strip()]

        if not lines:
            background_images.append(img_path)
            continue

        classes_in_image = set()
        for line in lines:
            cls_id = int(float(line.split()[0]))
            instance_counts[cls_id] += 1
            classes_in_image.add(cls_id)

        for cls_id in classes_in_image:
            images_per_class[cls_id].append(img_path)

    return instance_counts, images_per_class, background_images


train_dir = DATASET_PATH / "train"
instance_counts, images_per_class, background_images = analyze_dataset(train_dir, CLASS_NAMES)

print("=== Distribucion del set de entrenamiento ===")
print(f"\nInstancias (anotaciones) por clase:")
for cls_id, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {instance_counts[cls_id]}")

print(f"\nImagenes que contienen cada clase:")
for cls_id, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {len(images_per_class[cls_id])} imagenes")

print(f"\nImagenes background (sin anotaciones): {len(background_images)}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = CLASS_NAMES
img_counts = [len(images_per_class[i]) for i in range(len(CLASS_NAMES))]
inst_counts = [instance_counts[i] for i in range(len(CLASS_NAMES))]

colors = ["#2196F3", "#FF5722", "#4CAF50", "#9C27B0"]

axes[0].bar(names, img_counts, color=colors)
axes[0].set_title("Imagenes por clase (train)")
axes[0].set_ylabel("# Imagenes")
for i, v in enumerate(img_counts):
    axes[0].text(i, v + 0.5, str(v), ha="center", fontweight="bold")

axes[1].bar(names, inst_counts, color=colors)
axes[1].set_title("Instancias por clase (train)")
axes[1].set_ylabel("# Instancias")
for i, v in enumerate(inst_counts):
    axes[1].text(i, v + 0.5, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig(WORKING_DIR / "class_distribution.png", dpi=150)
plt.show()
print("Plot guardado.")

## 6. Manejo de imagenes background

In [ ]:
# Asegurar que todas las imagenes background tienen un .txt vacio
labels_dir = train_dir / "labels"
labels_dir.mkdir(exist_ok=True)

fixed_bg = 0
for img_path in background_images:
    label_path = labels_dir / (img_path.stem + ".txt")
    if not label_path.exists():
        label_path.touch()
        fixed_bg += 1

print(f"Labels vacios creados para {fixed_bg} imagenes background adicionales.")
print(f"Total imagenes background en train: {len(background_images)}")
print("(Las imagenes sin anotaciones ensenyan al modelo a no detectar cuando no hay vehiculos de emergencia.)")

## 7. Referencia al data.yaml

In [ ]:
# Usar el data.yaml original (augmentaciones y balanceo ya hechos en Roboflow)
DATA_YAML = DATASET_PATH / "data.yaml"

print(f"Usando data.yaml: {DATA_YAML}")
print("Contenido:")
with open(DATA_YAML) as f:
    print(f.read())

## 8. Configuracion y lanzamiento del entrenamiento

In [ ]:
project_dir = WORKING_DIR / "runs"
project_dir.mkdir(parents=True, exist_ok=True)
run_name = "emergency_vehicles_yolo26"

model = YOLO("yolo26m.pt")

print(f"Modelo: YOLO26m")
print(f"Dataset: {DATA_YAML}")
print(f"Clases: {CLASS_NAMES}")
print(f"Dispositivos: 2x T4 (device=[0, 1])")
print(f"Output: {project_dir / run_name}")

In [ ]:
results = model.train(
    data=str(DATA_YAML),
    epochs=200,
    imgsz=640,
    batch=48,
    device=[0, 1],

    # early stopping
    patience=40,

    save=True,
    project=str(project_dir),
    name=run_name,

    seed=42,
    cos_lr=True,
    lr0=0.01,
    lrf=0.1,
    warmup_epochs=3,

    # Augmentaciones online minimas (dataset ya augmentado en Roboflow)
    hsv_h=0.0,
    hsv_s=0.2,
    hsv_v=0.15,
    degrees=15.0,
    translate=0.1,
    scale=0.1,
    fliplr=0.5,
    mixup=0.0,
    erasing=0.0,

    mosaic=1.0,
    close_mosaic=15,
    copy_paste=0.0,
    cutmix=0.0,

    # Loss weights
    cls=1.0,
    box=7.5,
)

print("\nEntrenamiento completado.")
print(f"Mejor modelo: {project_dir / run_name / 'weights/best.pt'}")


## 9. Evaluacion — metricas en test set

In [ ]:
best_model_path = project_dir / run_name / "weights" / "best.pt"
eval_model = YOLO(str(best_model_path))

print("=== Evaluacion en validation set ===")
val_results = eval_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=640,
    device=0,
)

print("\n=== Evaluacion en test set ===")
test_results = eval_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    device=0,
)

# Mostrar metricas por clase
print("\n--- Metricas por clase (test set) ---")
print(f"{'Clase':<15} {'Precision':>10} {'Recall':>10} {'mAP@50':>10} {'mAP@50-95':>12}")
print("-" * 60)

if hasattr(test_results, 'box'):
    box = test_results.box
    for i, name in enumerate(CLASS_NAMES):
        try:
            p = box.p[i] if hasattr(box, 'p') else 0
            r = box.r[i] if hasattr(box, 'r') else 0
            ap50 = box.ap50[i] if hasattr(box, 'ap50') else 0
            ap = box.ap[i] if hasattr(box, 'ap') else 0
            print(f"{name:<15} {p:>10.4f} {r:>10.4f} {ap50:>10.4f} {ap:>12.4f}")
        except Exception:
            pass
    print("-" * 60)
    try:
        print(f"{'MEDIA':<15} {box.mp:>10.4f} {box.mr:>10.4f} {box.map50:>10.4f} {box.map:>12.4f}")
    except Exception:
        pass

# Verificacion del criterio de aceptacion (>=70% mAP@50)
try:
    map50 = test_results.box.map50
    criterio = "CUMPLIDO" if map50 >= 0.70 else "NO CUMPLIDO"
    print(f"\nCriterio de aceptacion (mAP@50 >= 0.70): {map50:.4f} -> {criterio}")
except Exception:
    print("\nNo se pudo obtener mAP@50 automaticamente. Revisar logs.")

## 10. Curvas de entrenamiento

In [ ]:
import pandas as pd

results_csv = project_dir / run_name / "results.csv"

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    metrics_to_plot = [
        ("train/box_loss", "Train Box Loss"),
        ("train/cls_loss", "Train Cls Loss"),
        ("val/box_loss", "Val Box Loss"),
        ("val/cls_loss", "Val Cls Loss"),
        ("metrics/mAP50(B)", "mAP@50"),
        ("metrics/mAP50-95(B)", "mAP@50-95"),
    ]

    for ax, (col, title) in zip(axes.flat, metrics_to_plot):
        if col in df.columns:
            ax.plot(df["epoch"], df[col], linewidth=2)
            ax.set_title(title)
            ax.set_xlabel("Epoch")
            ax.grid(True, alpha=0.3)
        else:
            ax.set_title(f"{title} (no disponible)")

    plt.suptitle("Curvas de entrenamiento — YOLO26m Emergency Vehicles", fontsize=14)
    plt.tight_layout()
    plt.savefig(WORKING_DIR / "training_curves.png", dpi=150)
    plt.show()
else:
    print("results.csv no encontrado. Verificar ruta del run.")

## 11. Visualizacion de predicciones

In [ ]:
test_images_dir = DATASET_PATH / "test" / "images"
test_images = (
    list(test_images_dir.glob("*.jpg"))
    + list(test_images_dir.glob("*.png"))
    + list(test_images_dir.glob("*.jpeg"))
)

# Seleccionar 8 imagenes aleatorias
sample_images = random.sample(test_images, min(8, len(test_images)))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
colors_map = {0: (255, 100, 100), 1: (100, 255, 100), 2: (100, 100, 255), 3: (200, 100, 200)}

for ax, img_path in zip(axes.flat, sample_images):
    pred = eval_model.predict(str(img_path), conf=0.25, verbose=False)[0]
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    for box in pred.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        color = colors_map.get(cls_id, (255, 255, 0))
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 2)
        label = f"{CLASS_NAMES[cls_id]} {conf:.2f}"
        cv2.putText(img_rgb, label, (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    ax.imshow(img_rgb)
    ax.set_title(img_path.name[:30], fontsize=8)
    ax.axis("off")

plt.suptitle("Predicciones (conf >= 0.25) — YOLO26m Emergency Vehicles", fontsize=13)
plt.tight_layout()
plt.savefig(WORKING_DIR / "predictions_sample.png", dpi=150)
plt.show()

## 12. Inferencia en imagenes background (verificacion de falsos positivos)

In [ ]:
# Buscar imagenes background en test set
test_labels_dir = DATASET_PATH / "test" / "labels"
test_bg_images = []

for img_path in test_images:
    lbl = test_labels_dir / (img_path.stem + ".txt")
    if not lbl.exists() or lbl.stat().st_size == 0:
        test_bg_images.append(img_path)

print(f"Imagenes background en test set: {len(test_bg_images)}")

if test_bg_images:
    sample_bg = random.sample(test_bg_images, min(4, len(test_bg_images)))
    fig, axes = plt.subplots(1, len(sample_bg), figsize=(5 * len(sample_bg), 5))
    if len(sample_bg) == 1:
        axes = [axes]

    false_positives = 0
    for ax, img_path in zip(axes, sample_bg):
        pred = eval_model.predict(str(img_path), conf=0.25, verbose=False)[0]
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        n_det = len(pred.boxes)
        if n_det > 0:
            false_positives += 1
        ax.imshow(img_rgb)
        ax.set_title(f"Detecciones: {n_det} (esperado: 0)", fontsize=9,
                     color="red" if n_det > 0 else "green")
        ax.axis("off")

    print(f"Falsos positivos en muestra: {false_positives}/{len(sample_bg)}")
    plt.suptitle("Imagenes background — verificacion de falsos positivos")
    plt.tight_layout()
    plt.savefig(WORKING_DIR / "background_check.png", dpi=150)
    plt.show()
else:
    print("No se encontraron imagenes background en el test set.")

## 13. Exportacion ONNX

In [ ]:
# Export best.pt → ONNX for Jetson Orin Nano deployment
onnx_model = YOLO(str(best_model_path))

onnx_path = onnx_model.export(
    format="onnx",
    imgsz=640,
    dynamic=False,   # fixed shape — required for TensorRT conversion on Jetson
    simplify=True,   # onnxsim graph optimization
    opset=11,        # ONNX opset 11 — broad runtime compatibility
)

print(f"ONNX exportado: {onnx_path}")

print(f"Tamaño: {Path(onnx_path).stat().st_size / 1e6:.2f} MB")


## 14. Guardar artefactos finales

In [ ]:
artifacts_dir = WORKING_DIR / "artifacts"
artifacts_dir.mkdir(exist_ok=True)

# Copiar modelos
run_weights = project_dir / run_name / "weights"
for weight_file in ["best.pt", "last.pt", "best.onnx"]:
    src = run_weights / weight_file
    if src.exists():
        shutil.copy(src, artifacts_dir / weight_file)
        print(f"Copiado: {weight_file}")

# Copiar metricas
for f in ["results.csv", "confusion_matrix.png", "results.png"]:
    src = project_dir / run_name / f
    if src.exists():
        shutil.copy(src, artifacts_dir / f)
        print(f"Copiado: {f}")

# Copiar plots generados
for plot in [
    "class_distribution.png",
    "training_curves.png",
    "predictions_sample.png",
    "background_check.png",
]:
    src = WORKING_DIR / plot
    if src.exists():
        shutil.copy(src, artifacts_dir / plot)
        print(f"Copiado: {plot}")

# Copiar data.yaml
shutil.copy(DATA_YAML, artifacts_dir / "data.yaml")

print("\n=== Artefactos finales ===")
for f in sorted(artifacts_dir.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:<35} {size_mb:.2f} MB")